In [1]:
import sys
from pathlib import Path

# Agregar la raíz del proyecto (Proyecto-Final-Henry) al path de Python
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ahora sí podemos importar desde src
from src.data_loader import cargar_datos
from src.preprocessing import procesar_etl, preparar_datos_modelo

# 1. Cargar datos
df = cargar_datos()

# 2. Probar preparación completa
X, y, preprocessor = preparar_datos_modelo(df)

[ETL] Registros duplicados eliminados: 125
[ETL] Sin valores nulos detectados.
[PREPROCESSING] Matriz procesada lista. Forma: (12205, 80)


# 🛠️ Registro de Decisiones Técnicas y de Arquitectura (ADR)
**Proyecto:** System Scoring & Recommendation Engine — Metric Mindset  
**Dataset:** Online Shoppers Purchasing Intention (UCI Repository)  

Este documento registra formalmente las decisiones de ingeniería de datos, preprocesamiento y diseño adoptadas en el proyecto, justificando el impacto técnico y de negocio de cada una.

---

## 1. Tratamiento de Registros Duplicados
* **Decisión:** Eliminar los **125 registros duplicados exactos** durante el pipeline de ETL (`src/preprocessing.py`).
* **Justificación:** Aunque en la etapa exploratoria (EDA) no se borraron preventivamente por representar sesiones de navegación anónimas, en la fase de ingeniería de datos se validó que la presencia de filas idénticas entre variables predictoras y variable objetivo introduce sesgos de sobreajuste (*overfitting*) en la evaluación cruzada.
* **Impacto:** El dataset pasó de **12.330 a 12.205 filas limpias**.

---

## 2. Tratamiento de Valores Faltantes (Nulos)
* **Decisión:** Implementar un mecanismo de imputación preventiva (Mediana para numéricas, Moda para categóricas) dentro de `procesar_etl`.
* **Justificación:** El dataset actual no presentó valores nulos en su estado base, pero el pipeline debe ser tolerante a fallos en tiempo de inferencia para procesar nuevos eventos de navegación en tiempo real.

---

## 3. Codificación de Variables Categóricas (One-Hot Encoding)
* **Decisión:** Aplicar `OneHotEncoder(handle_unknown='ignore')` a las variables categóricas (`Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType`, `Weekend`, `SpecialDay`).
* **Justificación:** Las variables como `OperatingSystems` o `Region` poseen un comportamiento estrictamente nominal, sin jerarquía ordinal explícita. Para evitar que los algoritmos interpreten relaciones de orden erróneas, se transformaron en vectores binarios (*dummy variables*).
* **Compatibilidad:** Se diseñó un bloque de excepción dinámica (`try/except`) para soportar diferencias entre versiones de Scikit-Learn (`sparse` vs `sparse_output`).

---

## 4. Escalado de Métricas Web Numéricas
* **Decisión:** Aplicar `StandardScaler` sobre todas las métricas continuas y de duración (`Administrative`, `ProductRelated_Duration`, `PageValues`, etc.).
* **Justificación:** Las métricas de tiempo y número de páginas visitadas presentan órdenes de magnitud muy dispares (ej. `ProductRelated_Duration` en miles de segundos vs `BounceRates` en decimales entre 0 y 1). El escalado homogeniza las magnitudes para algoritmos basados en distancias y gradientes.

---

## 5. Transformación del Target (`Revenue`)
* **Decisión:** Convertir la variable booleana `Revenue` (`True`/`False`) a entero binario (`1`/`0`).
* **Justificación:** Estandariza la variable objetivo para la compatibilidad con funciones de pérdida en modelos predictivos (Scikit-Learn, XGBoost, LightGBM) y facilita el cálculo directo de métricas como *Precision*, *Recall* y *ROC-AUC*.

---

## 6. Manejo Futuro del Desbalance de Clases (*Sparsity*)
* **Decisión:** Mantener el desbalance natural (~15.5% clase positiva) en el preprocesamiento base y aplicar técnicas de balanceo (SMOTE / `class_weight='balanced'`) **únicamente dentro del flujo de modelado**.
* **Justificación:** Evita la contaminación de datos (*data leakage*) antes de la división de entrenamiento y prueba (*train/test split*).